<a href="https://colab.research.google.com/github/nicholastimmann-cyber/Travel_Tide_CustomerSegmentation/blob/main/detecting_canceled_trips.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
directory = "/content/drive/MyDrive/Colab Notebooks"


df_session = pd.read_csv(f'{directory}/session_base_cleaned.csv')

for col in ['session_start', 'session_end', 'departure_time', 'return_time']:
    df_session[col] = pd.to_datetime(df_session[col],format='mixed')

print(df_session.shape)
df_session.head()

(49211, 42)


,session_id,user_id,trip_id,session_start,session_end,page_clicks,flight_discount,flight_discount_amount,hotel_discount,hotel_discount_amount,...,destination_airport_lat,destination_airport_lon,base_fare_usd,hotel_name,nights,rooms,check_in_time,check_out_time,hotel_price_per_room_night_usd,session_duration
0,167852-1b6e3f994cd34bacb88f18c91c7049ed,167852,NaN,2023-03-14 06:21:00,2023-03-14 06:21:24,3,False,NaN,True,0.10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.0
1,340166-fcf78962180d49c8a039e3943560b0b2,340166,NaN,2023-03-14 14:32:00,2023-03-14 14:35:36,29,False,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,216.0
2,374341-6d5602a6f09b460bb8e8f6ab1d22b2f0,374341,NaN,2023-03-14 21:28:00,2023-03-14 21:28:23,3,False,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.0
3,494484-7681e71f367e419c8cfc3a4d1433a763,494484,NaN,2023-03-14 20:51:00,2023-03-14 20:52:17,10,True,0.1,True,0.05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77.0
4,532361-27b44abf967a460cbf2ec773ed207389,532361,NaN,2023-03-14 21:34:00,2023-03-14 21:34:22,3,False,NaN,False,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0


In [ ]:
#Amount of cancelled and not cancelled trips
df_session.groupby("cancellation").size().reset_index(name="count")

,cancellation,count
0,False,48601
1,True,610


In [ ]:
#find canceled sessions
import pandas as pd
from sqlalchemy import create_engine

# Define your PostgreSQL connection parameters
username = 'Test'
password = 'bQNxVzJL4g6u'
host = 'ep-noisy-flower-846766.us-east-2.aws.neon.tech'        # or your DB host
port = '5432'             # default PostgreSQL port
database = 'TravelTide'

# Create connection string using SQLAlchemy
conn_str = f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}'
engine = create_engine(conn_str)

query = '''
SELECT DISTINCT trip_id
FROM sessions
WHERE cancellation = TRUE
'''

canceled_trip_ids = set(pd.read_sql(query, engine)['trip_id'].values)
print('Number of canceled trips (total)', len(canceled_trip_ids))
engine.dispose()

Number of canceled trips (total) 90670


Checking if the canceld trips are in the list of all canceled trips

In [ ]:
trip_id_example = df_session[df_session['cancellation'] == True]['trip_id'].iloc[0]
assert trip_id_example in canceled_trip_ids

We have in Total 600 canceld Trips after our Cohor Filtering

In [ ]:
df_canceled_trips = df_session[df_session['trip_id'].isin(canceled_trip_ids)]
df_canceled_trips = df_canceled_trips.drop_duplicates(subset=['trip_id'])
print(df_canceled_trips.shape[0])

610


We have 15,000 trips that belong to Sessions without canceled trips

In [ ]:
df_not_canceled_trips = df_session.dropna(subset = ['trip_id'])
df_not_canceled_trips = df_not_canceled_trips[~df_not_canceled_trips['trip_id'].isin(canceled_trip_ids)]
print(df_not_canceled_trips.shape[0])

15489


We drop any duplicate sessions

In [ ]:
df_not_canceled_trips = df_not_canceled_trips.drop_duplicates(subset=['trip_id'])
print(df_not_canceled_trips.shape[0])

15489


In [ ]:
print(df_not_canceled_trips.isnull().sum())

session_id                            0
user_id                               0
trip_id                               0
session_start                         0
session_end                           0
page_clicks                           0
flight_discount                       0
flight_discount_amount            13180
hotel_discount                        0
hotel_discount_amount             13347
flight_booked                         0
hotel_booked                          0
cancellation                          0
birthdate                             0
gender                                0
married                               0
has_children                          0
home_country                          0
home_city                             0
home_airport                          0
home_airport_lat                      0
home_airport_lon                      0
sign_up_date                          0
origin_airport                     2332
destination                        2332


In [ ]:
#Creation of t
df_not_canceled_trips.to_csv(f'{directory}/not_canceled_trips.csv', index=False)